# 1. Create Bronze Table

#### customers_bronze

- 購入が完了した顧客一覧

In [0]:
%sql
SELECT *
FROM read_files(
    "/Volumes/databricks_simulated_retail_customer_data/v01/source_files/customers.csv",
    format => 'csv',
    sep => ','
)
LIMIT 10;

|customer_id|tax_id|tax_code|customer_name|state|city|postcode|street|number|unit|region|district|lon|lat|ship_to_address|valid_from|valid_to|units_purchased|loyalty_segment|_rescued_data|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|11123757|NULL|NULL|SMITH,  SHIRLEY|IN|BREMEN|46506.0|N CENTER ST|521.0|NULL|Indiana|50.0|-86.1465825|41.4507625|IN, 46506.0, N CENTER ST, 521.0|1532824233|1548137353|34|3|null|
|30585978|NULL|NULL|STEPHENS,  GERALDINE M|OR|ADDRESS|0|NO SITUS|NULL|NULL|NULL|NULL|-122.1055158|45.374317|OR, 0, NO SITUS, nan|1523100473|NULL|18|3|null|
|349822|NULL|NULL|GUZMAN,  CARMEN|VA|VIENNA|22181|HILL RD|2860|NULL|VA|NULL|-77.2941261|38.88303270000001|VA, 22181, HILL RD, 2860|1522922493|NULL|5|0|null|
|27652636|NULL|NULL|HASSETT,  PATRICK J|WI|VILLAGE OF NASHOTAH|53058.0|IVY LANE|W333N 5591|NULL|NULL|NULL|-88.40951700000002|43.1213789|WI, 53058.0, IVY LANE, W333N 5591|1531834357|1558052195|7|1|null|
|14437343|NULL|NULL|HENTZ,  DIANA L|OH|COLUMBUS|43228.0|ALLIANCE WAY|5706|NULL|OH|FRA|-83.158438|39.97821810000001|OH, 43228.0, ALLIANCE WAY, 5706|1517227530|NULL|0|0|null|
|20441596|NULL|NULL|TIRADO,  MARCO A|NY|Otselic|13072|County Road 16|2792|NULL|NY|Chenango|-75.7505808|42.7172722|NY, 13072, County Road 16, 2792|1519335250|NULL|24|3|null|
|5945686|NULL|NULL|SKORA,  BRIAN S|MI|NULL|48205.0|E 8 MILE RD|16414.0|NULL|NULL|NULL|-82.950874|42.4499233|MI, 48205.0, E 8 MILE RD, 16414.0|1518988242|NULL|7|1|null|
|5385771|NULL|NULL|SLAWEK,  DEAN J|PA|NULL|19147-3204|FITZWATER ST|328|NULL|NULL|NULL|-75.14920550000002|39.9389473|PA, 19147-3204, FITZWATER ST, 328|1518239268|NULL|18|3|null|
|1427940|NULL|NULL|REAVES,  LIONEL C|VA|HOT SPRINGS|24445.0|HOT SPRINGS RD|6419.0|NULL|NULL|NULL|-79.90497859999998|37.8949737|VA, 24445.0, HOT SPRINGS RD, 6419.0|1529087690|NULL|10|2|null|
|10457387|NULL|NULL|BONGIOVANNI,  KELLY M|IN|VINCENNES|47591|JERRY ST|2006.0|NULL|Indiana|42.0|-87.519002|38.662178|IN, 47591, JERRY ST, 2006.0|1535887733|NULL|9|2|null|

- customers_bronzeテーブルを作成する

In [0]:
%sql
CREATE OR REPLACE TABLE customers_bronze
AS
SELECT *,
  current_timestamp() as ingestion_timestamp
FROM read_files(
    "/Volumes/databricks_simulated_retail_customer_data/v01/source_files/customers.csv",
    format => 'csv',
    sep => ',',
    mode => 'FAILFAST',
    nullValue => 'NULL'
);

-- 作成したテーブルの確認
SELECT * FROM customers_bronze LIMIT 5;

In [0]:
%sql
DESCRIBE EXTENDED customers_bronze;


--------

### sales

In [0]:
%sql
SELECT *
FROM read_files(
    "/Volumes/databricks_simulated_retail_customer_data/v01/source_files/sales.csv",
    header => true,
    format => 'csv',
    sep => ',',
    escape => '"'

);

- sales_bronzeテーブルを作成する

In [0]:
%sql
CREATE OR REPLACE TABLE sales_bronze
AS
SELECT 
  customer_id,
  customer_name,
  product_name,
  order_date,
  product_category,
  REPLACE(product, '""', '"') AS product, -- エスケープがおかしいので修正
  total_price,
  current_timestamp() AS ingestion_timestamp
FROM read_files(
    "/Volumes/databricks_simulated_retail_customer_data/v01/source_files/sales.csv",
    format => 'csv',
    sep => ',',
    escape => '"',
    mode => 'FAILFAST',
    nullValue => 'NULL'
);

-- 作成したテーブルの確認
SELECT * FROM sales_bronze LIMIT 5;

In [0]:
%sql
DESCRIBE EXTENDED sales_bronze;

--------------

### sales_orders

In [0]:
%sql
SELECT 
    *
FROM read_files(
    "/Volumes/databricks_simulated_retail_customer_data/v01/source_files/sales_orders.csv",
    format => "csv",
    header => true,
    sep => ",",
    escape => '"'
)
LIMIT 10;

- sales_orders_bronzeテーブルを作成する

In [0]:
%sql
CREATE OR REPLACE TABLE sales_orders_bronze
AS
SELECT *,
  current_timestamp() as ingestion_timestamp
FROM read_files(
    "/Volumes/databricks_simulated_retail_customer_data/v01/source_files/sales_orders.csv",
    format => 'csv',
    sep => ',',
    escape => '"',
    mode => 'FAILFAST',
    nullValue => 'NULL'
);

-- 作成したテーブルの確認
SELECT * FROM sales_orders_bronze LIMIT 5;

In [0]:
%sql
DESCRIBE EXTENDED sales_orders_bronze;

----------------------